# Feature Engineering and Data Splitting

This notebook prepares the motor temperature dataset for machine learning.

The main steps are:
1. Load the cleaned dataset
2. Create physically meaningful engineered features
3. Separate features and target
4. Split data by motor profile to prevent data leakage
5. Prepare the data for model training

In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../data/raw/measures_v2.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (1330816, 13)


In [2]:
print("Columns:")
print(df.columns.tolist())

print("\nNumber of profiles:", df["profile_id"].nunique())

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Columns:
['u_q', 'coolant', 'stator_winding', 'u_d', 'stator_tooth', 'motor_speed', 'i_d', 'i_q', 'pm', 'stator_yoke', 'ambient', 'torque', 'profile_id']

Number of profiles: 69

Missing values:
0

Duplicate rows:
0


## 1. Engineering Physically Meaningful Features

Additional features are derived from the electrical and mechanical
parameters of the motor.

- Current magnitude combines the d-axis and q-axis current.
- Voltage magnitude combines the d-axis and q-axis voltage.
- Mechanical power is calculated from torque and rotational speed.

These features provide the model with physically meaningful
representations of motor operating conditions.

In [3]:
# Current magnitude
df["current_magnitude"] = np.sqrt(
    df["i_d"]**2 + df["i_q"]**2
)

# Voltage magnitude
df["voltage_magnitude"] = np.sqrt(
    df["u_d"]**2 + df["u_q"]**2
)

# Mechanical power (Watts)
df["mechanical_power"] = (
    df["torque"] * (2 * np.pi * df["motor_speed"] / 60)
)

print("Engineered features created successfully!")

print("\nNew columns:")
print([
    "current_magnitude",
    "voltage_magnitude",
    "mechanical_power"
])

Engineered features created successfully!

New columns:
['current_magnitude', 'voltage_magnitude', 'mechanical_power']


In [4]:
df[
    [
        "current_magnitude",
        "voltage_magnitude",
        "mechanical_power"
    ]
].describe()

,current_magnitude,voltage_magnitude,mechanical_power
count,1.330816e+06,1.330816e+06,1.330816e+06
mean,1.075446e+02,8.164042e+01,6.513603e+03
std,8.526254e+01,5.333507e+01,1.858097e+04
min,3.922213e-05,1.732221e-06,-5.526779e+04
25%,1.182819e+01,1.890028e+01,-2.693797e+01
50%,1.106200e+02,1.077091e+02,1.027488e+03
75%,1.721419e+02,1.307828e+02,2.043910e+04
max,3.615153e+02,1.342641e+02,4.961819e+04


In [5]:
print("Missing values:")
print(df[
    [
        "current_magnitude",
        "voltage_magnitude",
        "mechanical_power"
    ]
].isnull().sum())

print("\nInfinite values:")
print(
    np.isinf(
        df[
            [
                "current_magnitude",
                "voltage_magnitude",
                "mechanical_power"
            ]
        ]
    ).sum()
)

Missing values:
current_magnitude    0
voltage_magnitude    0
mechanical_power     0
dtype: int64

Infinite values:
current_magnitude    0
voltage_magnitude    0
mechanical_power     0
dtype: int64


## 2. Feature and Target Definition

The target variable is `stator_winding`, representing the stator
winding temperature that the model must predict.

`profile_id` is excluded from the predictive features because it
identifies the motor profile rather than representing an operating
condition. Including it could allow the model to memorize profile
specific behavior.

In [6]:
target = "stator_winding"

feature_columns = [
    "u_q",
    "coolant",
    "u_d",
    "stator_tooth",
    "motor_speed",
    "i_d",
    "i_q",
    "pm",
    "stator_yoke",
    "ambient",
    "torque",
    "current_magnitude",
    "voltage_magnitude",
    "mechanical_power"
]

X = df[feature_columns]
y = df[target]

groups = df["profile_id"]

print("Number of features:", len(feature_columns))
print("Features:")
print(feature_columns)

print("\nTarget:", target)
print("Rows:", len(X))

Number of features: 14
Features:
['u_q', 'coolant', 'u_d', 'stator_tooth', 'motor_speed', 'i_d', 'i_q', 'pm', 'stator_yoke', 'ambient', 'torque', 'current_magnitude', 'voltage_magnitude', 'mechanical_power']

Target: stator_winding
Rows: 1330816


## 3. Profile-Based Train/Validation/Test Split

The dataset contains 69 motor profiles with multiple sequential
observations per profile.

To prevent data leakage, entire profiles are assigned to only one
dataset. No profile will occur simultaneously in the training,
validation, and test sets.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

# First split: 70% training, 30% temporary
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()

X_temp = X.iloc[temp_idx].copy()
y_temp = y.iloc[temp_idx].copy()

groups_temp = groups.iloc[temp_idx]

print("Training rows:", len(X_train))
print("Temporary rows:", len(X_temp))

print("\nTraining profiles:", groups.iloc[train_idx].nunique())
print("Temporary profiles:", groups_temp.nunique())

Training rows: 879860
Temporary rows: 450956

Training profiles: 48
Temporary profiles: 21


In [8]:
# Split temporary data into validation and test
gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss2.split(X_temp, y_temp, groups=groups_temp)
)

X_val = X_temp.iloc[val_idx].copy()
y_val = y_temp.iloc[val_idx].copy()

X_test = X_temp.iloc[test_idx].copy()
y_test = y_temp.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))

print("\nTraining profiles:", groups.iloc[train_idx].nunique())
print("Validation profiles:", groups_temp.iloc[val_idx].nunique())
print("Test profiles:", groups_temp.iloc[test_idx].nunique())

Training rows: 879860
Validation rows: 211545
Test rows: 239411

Training profiles: 48
Validation profiles: 10
Test profiles: 11


In [9]:
train_profiles = set(groups.iloc[train_idx])
val_profiles = set(groups_temp.iloc[val_idx])
test_profiles = set(groups_temp.iloc[test_idx])

print("Train ∩ Validation:", train_profiles & val_profiles)
print("Train ∩ Test:", train_profiles & test_profiles)
print("Validation ∩ Test:", val_profiles & test_profiles)

Train ∩ Validation: set()
Train ∩ Test: set()
Validation ∩ Test: set()
